<a href="https://colab.research.google.com/github/mariogaudio-bot/Progetto-Reti/blob/main/progetto1_TTR_notebook_di_partenza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Progetto 1 — Classificare le varianti di TTR con i Graph Transformer
**Corso Graph Transformers — notebook di partenza comune alle 4 tracce**

| Traccia | Domanda di ricerca |
|---|---|
| **A — Tetramero** | Rappresentare la proteina come *tetramero* (4 × residui, archi tipizzati) migliora la classificazione rispetto al monomero? |
| **B — Fisica + apprendimento** | Feature fisiche (flessibilità, accessibilità, sito di clivaggio, canale della T4, dinamica molecolare) aiutano a generalizzare a **siti mai visti**? |
| **C — Più compiti** | Imparare insieme patogenicità e **fenotipo** (neuropatico / cardiaco) migliora il compito principale? |
| **D — VUS e calibrazione** | Come si confronta il GT con **AlphaMissense** e ESM-2? Che forza di evidenza ACMG/AMP può dare? Quali VUS meritano attenzione? |

### Regole comuni (valgono per tutti i gruppi)
1. **Stesso dataset congelato** (`ttr_dataset_v1.csv`) e **stessi 5 fold** per posizione: non rigenerateli.
2. Ogni risultato passa da `valuta(...)`, che scrive in `risultati.csv` (AUROC con IC 95% bootstrap, AUPRC, MCC).
3. Ogni traccia si confronta con le **tre baseline** calcolate in questo notebook.
4. Niente scelte fatte guardando il test: gli iperparametri si scelgono con una validazione **interna** al training.
5. Consegna: notebook eseguito, `risultati.csv`, report di 4 pagine (template fornito), presentazione di 10 minuti.

Eseguite **tutta la parte comune** (sezioni 0–2), poi andate alla sezione della vostra traccia. ⏱️ La parte comune richiede ~15 min con GPU.

In [ ]:
!pip install -q transformers torch_geometric

## 0. Dati (come nella lezione 4)

In [ ]:
import re, json, time, urllib.request, urllib.parse
import numpy as np, pandas as pd

def get(url, data=None, headers=None, timeout=90):
    h = {'User-Agent': 'Mozilla/5.0 (corso Graph Transformers)'}
    h.update(headers or {})
    return urllib.request.urlopen(urllib.request.Request(url, data=data, headers=h), timeout=timeout).read()

fasta = get("https://rest.uniprot.org/uniprotkb/P02766.fasta").decode()
SEQ = "".join(fasta.splitlines()[1:]).strip()
SIGNAL = 20                                   # peptide segnale: residui 1-20 del precursore
MATURE = SEQ[SIGNAL:]
print(fasta.splitlines()[0])
print(f"precursore: {len(SEQ)} aa | proteina matura: {len(MATURE)} aa")
print("segnale :", SEQ[:SIGNAL]); print("matura  :", MATURE)

In [ ]:
AA3 = {'Ala': 'A', 'Arg': 'R', 'Asn': 'N', 'Asp': 'D', 'Cys': 'C', 'Gln': 'Q', 'Glu': 'E', 'Gly': 'G',
       'His': 'H', 'Ile': 'I', 'Leu': 'L', 'Lys': 'K', 'Met': 'M', 'Phe': 'F', 'Pro': 'P', 'Ser': 'S',
       'Thr': 'T', 'Trp': 'W', 'Tyr': 'Y', 'Val': 'V'}
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"

def eutils(tool, **params):
    params.setdefault('retmode', 'json')
    out = json.loads(get(EUTILS + tool + "?" + urllib.parse.urlencode(params)))
    time.sleep(0.35)                                   # max ~3 richieste/s senza API key
    return out

ids = eutils('esearch.fcgi', db='clinvar', term='TTR[gene] AND single_gene[prop]', retmax=5000)['esearchresult']['idlist']
docs = {}
for i in range(0, len(ids), 200):
    r = eutils('esummary.fcgi', db='clinvar', id=",".join(ids[i:i + 200]))['result']
    docs.update({u: r[u] for u in r['uids']})
print(len(docs), "record ClinVar per TTR")

def classification(d):
    for key in ['germline_classification', 'clinical_significance']:      # nome nuovo e vecchio del campo
        v = d.get(key)
        if isinstance(v, dict) and v.get('description'):
            return v['description'], v.get('review_status', '')
    return '', ''

def parse_clinvar(docs):
    rows = []
    for uid, d in docs.items():
        m = re.search(r'\(TTR\)[^()]*\(p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})\)', d.get('title', ''))
        if not m:
            continue
        wt, pos, mt = AA3.get(m.group(1)), int(m.group(2)), AA3.get(m.group(3))
        if not wt or not mt or wt == mt:
            continue                                        # solo missense
        c, rev = classification(d)
        rows.append(dict(clinvar_id=uid, wt=wt, pos=pos, mt=mt, classificazione=c, revisione=rev))
    return pd.DataFrame(rows)

def to_label(c):
    c = c.lower()
    if not c or 'conflicting' in c or 'uncertain' in c:
        return np.nan
    if 'pathogenic' in c and 'benign' not in c:
        return 1.0
    if ('benign' in c or 'protective' in c) and 'pathogenic' not in c:
        return 0.0
    return np.nan

cv = parse_clinvar(docs)
cv['label'] = cv.classificazione.map(to_label)
cv['fonte'] = 'ClinVar'
cv = cv.sort_values('label', na_position='last').drop_duplicates(['pos', 'wt', 'mt']).reset_index(drop=True)

In [ ]:
# Controllo di sanità: il residuo "wt" di ClinVar coincide con la sequenza UniProt?
cv['wt_ok'] = [SEQ[p - 1] == w if 1 <= p <= len(SEQ) else False for p, w in zip(cv.pos, cv.wt)]
print(f"wt coerenti con UniProt: {cv.wt_ok.mean():.1%}")
cv = cv[cv.wt_ok].drop(columns='wt_ok')
cv['nome_hgvs'] = 'p.' + cv.wt + cv.pos.astype(str) + cv.mt
cv['nome_classico'] = [f"{w}{p - SIGNAL}{m}" if p > SIGNAL else '(segnale)' for w, p, m in zip(cv.wt, cv.pos, cv.mt)]
print(cv.classificazione.value_counts().head(10))
print("\netichette: ", cv.label.value_counts(dropna=False).rename({1.0: 'patogenica', 0.0: 'benigna'}).to_dict())

FAMOSE = {'V30M': 'neuropatia, Portogallo/Svezia/Giappone', 'V122I': 'cardiomiopatia, ~3-4% afro-discendenti',
          'T60A': 'Irlanda, cardio-neuro', 'L55P': 'forma aggressiva, precoce', 'T119M': 'protettiva (stabilizza il tetramero)'}
display(cv[cv.nome_classico.isin(FAMOSE)][['nome_classico', 'nome_hgvs', 'classificazione', 'revisione']])

In [ ]:
# gnomAD: varianti missense osservate nella popolazione generale (opzionale)
USE_GNOMAD, AF_MIN = True, 1e-4
gn = pd.DataFrame()
if USE_GNOMAD:
    q = """query($g: String!) { gene(gene_symbol: $g, reference_genome: GRCh38) {
             variants(dataset: gnomad_r4) { variant_id hgvsp consequence exome { ac an } genome { ac an } } } }"""
    try:
        r = json.loads(get("https://gnomad.broadinstitute.org/api",
                           data=json.dumps({'query': q, 'variables': {'g': 'TTR'}}).encode(),
                           headers={'Content-Type': 'application/json'}))
        rows = []
        for v in r['data']['gene']['variants']:
            m = re.match(r'p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})$', v.get('hgvsp') or '')
            if not m or 'missense' not in (v.get('consequence') or ''):
                continue
            ac = sum((v.get(k) or {}).get('ac', 0) or 0 for k in ['exome', 'genome'])
            an = sum((v.get(k) or {}).get('an', 0) or 0 for k in ['exome', 'genome'])
            rows.append(dict(wt=AA3[m.group(1)], pos=int(m.group(2)), mt=AA3[m.group(3)], af=ac / an if an else 0))
        gn = pd.DataFrame(rows).groupby(['pos', 'wt', 'mt'], as_index=False).af.max()
        print(len(gn), "missense in gnomAD")
    except Exception as e:
        print("gnomAD non raggiungibile:", e)

if len(gn):
    merged = gn.merge(cv[['pos', 'wt', 'mt', 'label', 'classificazione']], on=['pos', 'wt', 'mt'], how='left')
    print("varianti PATOGENICHE in ClinVar ma frequenti in gnomAD (AF ≥ 1e-4):")
    display(merged[(merged.label == 1) & (merged.af >= AF_MIN)].sort_values('af', ascending=False).head(8))
    extra = merged[merged.label.isna() & (merged.af >= AF_MIN) & merged.classificazione.isna()].copy()
    extra['label'], extra['fonte'], extra['classificazione'] = 0.0, 'gnomAD (presunta benigna)', ''
    extra['nome_hgvs'] = 'p.' + extra.wt + extra.pos.astype(str) + extra.mt
    extra['nome_classico'] = [f"{w}{p - SIGNAL}{m}" if p > SIGNAL else '(segnale)' for w, p, m in zip(extra.wt, extra.pos, extra.mt)]
    extra = extra[[SEQ[p - 1] == w for p, w in zip(extra.pos, extra.wt)]]
    print(f"aggiunte {len(extra)} varianti presunte benigne da gnomAD (AF ≥ {AF_MIN}, assenti da ClinVar)")
    variants = pd.concat([cv, extra.drop(columns=['af'])], ignore_index=True)
else:
    variants = cv.copy()
labeled = variants.dropna(subset=['label']).reset_index(drop=True)
print("\nDATASET ETICHETTATO:", labeled.groupby(['fonte', 'label']).size().to_dict())
if (labeled.label == 0).sum() < 15:
    print("⚠️ pochissime varianti benigne: i risultati saranno molto instabili (prova AF_MIN = 5e-5)")

### 0.1 Dataset congelato e fold
ClinVar cambia ogni settimana: senza congelare i dati, due gruppi che scaricano in giorni diversi non sarebbero confrontabili.

In [ ]:
# ============ DATASET CONGELATO: tutti i gruppi lavorano sugli STESSI dati e sugli STESSI fold ============
# Il docente esegue questa cella una volta (DATASET_URL vuoto), scarica 'ttr_dataset_v1.csv'
# e lo condivide (Drive/GitHub). Gli studenti incollano qui il link o caricano il file.
import os, io
from sklearn.model_selection import StratifiedGroupKFold
DATASET_URL = ""                      # es. link "raw" a ttr_dataset_v1.csv
DATASET_FILE = "ttr_dataset_v1.csv"
SEED_FOLD = 2026

if DATASET_URL:
    open(DATASET_FILE, 'wb').write(get(DATASET_URL))
if os.path.exists(DATASET_FILE):
    labeled = pd.read_csv(DATASET_FILE)
    print(f"dataset congelato caricato: {len(labeled)} varianti")
else:
    labeled = labeled[labeled.pos > SIGNAL].reset_index(drop=True)
    labeled['fold'] = -1
    sgkf = StratifiedGroupKFold(5, shuffle=True, random_state=SEED_FOLD)
    for f, (_, te) in enumerate(sgkf.split(labeled, labeled.label.astype(int), labeled.pos)):
        labeled.loc[te, 'fold'] = f
    labeled['data_estrazione'] = time.strftime('%Y-%m-%d')
    labeled.to_csv(DATASET_FILE, index=False)
    print(f"⚠️ dataset NUOVO creato e salvato in {DATASET_FILE}: il docente lo condivide con tutti i gruppi")
print(labeled.groupby(['fold', 'label']).size().unstack())

In [ ]:
def parse_ca(pdb_text):
    """{(catena, numero): (coordinate CA, B-factor)}; ignora conformazioni alternative oltre la prima"""
    out = {}
    for l in pdb_text.splitlines():
        if l.startswith('ATOM') and l[12:16].strip() == 'CA' and l[16] in ' A':
            key = (l[21], int(l[22:26]))
            if key not in out:
                out[key] = (np.array([float(l[30:38]), float(l[38:46]), float(l[46:54])]), float(l[60:66]))
    return out

meta = json.loads(get("https://alphafold.ebi.ac.uk/api/prediction/P02766"))[0]
AF_TEXT = get(meta['pdbUrl']).decode()
af = parse_ca(AF_TEXT)
xtal = parse_ca(get("https://files.rcsb.org/download/1F41.pdb").decode())
print("AlphaFold:", meta.get('modelEntityId', meta['pdbUrl'].split('/')[-1]), "|", len(af), "residui")
print("PDB 1F41: catene", sorted({c for c, _ in xtal}), "|", len(xtal), "residui")

# Nodi = residui della proteina MATURA (precursore 21-147), numerazione del precursore = HGVS
POS = np.array([p for p in range(SIGNAL + 1, len(SEQ) + 1) if ('A', p) in af])
XYZ = np.stack([af[('A', p)][0] for p in POS])
PLDDT = np.array([af[('A', p)][1] for p in POS]) / 100
D = np.linalg.norm(XYZ[:, None] - XYZ[None], axis=-1)
CUTOFF = 8.0
A = (D < CUTOFF) & ~np.eye(len(POS), dtype=bool)
EDGE_INDEX = np.array(np.nonzero(A))
DEG = A.sum(1)

# Interfaccia monomero-monomero (dimero A-B nell'unità asimmetrica di 1F41, numerazione matura)
chainA = {n: c for (ch, n), (c, _) in xtal.items() if ch == 'A'}
chainB = np.array([c for (ch, n), (c, _) in xtal.items() if ch == 'B'])
INTERFACE = np.array([bool(len(chainB)) and (n - SIGNAL) in chainA and
                      np.min(np.linalg.norm(chainB - chainA[n - SIGNAL], axis=1)) < CUTOFF for n in POS])
P2I = {p: i for i, p in enumerate(POS)}                 # posizione HGVS -> indice del nodo
print(f"nodi: {len(POS)} | archi (orientati): {EDGE_INDEX.shape[1]} | residui all'interfaccia A-B: {INTERFACE.sum()}")

In [ ]:
import torch
from transformers import AutoTokenizer, EsmForMaskedLM
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ESM_NAME = "facebook/esm2_t12_35M_UR50D"      # più grandi: esm2_t30_150M_UR50D, esm2_t33_650M_UR50D (GPU)
tok = AutoTokenizer.from_pretrained(ESM_NAME)
try:
    esm = EsmForMaskedLM.from_pretrained(ESM_NAME, attn_implementation="eager")
except TypeError:
    esm = EsmForMaskedLM.from_pretrained(ESM_NAME)
esm = esm.to(device).eval()
AAS = 'ACDEFGHIKLMNPQRSTVWY'
AA_IDS = [tok.convert_tokens_to_ids(a) for a in AAS]
enc = tok(SEQ, return_tensors='pt').to(device)          # [CLS] + 147 aa + [EOS]: il residuo i è al token i
print(ESM_NAME, "|", sum(p.numel() for p in esm.parameters()) / 1e6, "M parametri | token:", enc.input_ids.shape[1])

In [ ]:
def masked_marginals(seq_ids, bs=16):
    """LLR[i, a] = log p(a | contesto, i mascherato) - log p(wt | ...). Negativo = sostituzione 'innaturale'."""
    L = seq_ids.shape[1] - 2
    LLR = np.zeros((L, 20), dtype=np.float32)
    for s in range(0, L, bs):
        pos = list(range(s, min(s + bs, L)))
        batch = seq_ids.repeat(len(pos), 1)
        for b, i in enumerate(pos):
            batch[b, i + 1] = tok.mask_token_id
        with torch.no_grad():
            logits = esm(input_ids=batch).logits
        for b, i in enumerate(pos):
            lp = logits[b, i + 1].log_softmax(-1)
            LLR[i] = (lp[AA_IDS] - lp[seq_ids[0, i + 1]]).cpu().numpy()
    return LLR

LLR = masked_marginals(enc.input_ids)
with torch.no_grad():
    H_ESM = esm(**enc, output_hidden_states=True).hidden_states[-1][0, 1:-1].cpu().numpy()   # (147, d)
print("LLR:", LLR.shape, "| embedding per residuo:", H_ESM.shape)

In [ ]:
from sklearn.decomposition import PCA
KD = dict(A=1.8, R=-4.5, N=-3.5, D=-3.5, C=2.5, Q=-3.5, E=-3.5, G=-0.4, H=-3.2, I=4.5,
          L=3.8, K=-3.9, M=1.9, F=2.8, P=-1.6, S=-0.8, T=-0.7, W=-0.9, Y=-1.3, V=4.2)   # idrofobicità Kyte-Doolittle
N_NODES = len(POS)
WT = np.array([SEQ[p - 1] for p in POS])
onehot = lambda s: np.eye(20)[[AAS.index(a) for a in s]]
z = lambda v: (v - v.mean(0)) / (v.std(0) + 1e-6)

emb = z(PCA(16, random_state=0).fit_transform(H_ESM[POS - 1]))           # embedding ESM per residuo
site_tol = z(LLR[POS - 1].mean(1))                                         # tolleranza media del sito
NODE_BASE = np.hstack([onehot(WT), emb, PLDDT[:, None], INTERFACE[:, None].astype(float),
                       (DEG / DEG.max())[:, None], ((POS - POS.min()) / np.ptp(POS))[:, None],
                       site_tol[:, None]]).astype(np.float32)
NODE_NAMES = [f'wt_{a}' for a in AAS] + [f'esm_{i}' for i in range(16)] + ['pLDDT', 'interfaccia', 'grado', 'posizione', 'tolleranza_sito']

def variant_x(pos, mt):
    """Feature dei nodi per una variante: base + [flag mutato | aa mutante one-hot | LLR | Δidrofobicità]"""
    i = P2I[pos]
    v = np.zeros((N_NODES, 23), dtype=np.float32)
    v[i, 0] = 1; v[i, 1 + AAS.index(mt)] = 1
    v[i, 21] = LLR[pos - 1, AAS.index(mt)] / 5
    v[i, 22] = (KD[mt] - KD[SEQ[pos - 1]]) / 5
    return np.hstack([NODE_BASE, v])

def tab_features(pos, mt):
    """Le stesse informazioni del sito mutato, in forma tabellare (per le baseline senza grafo)"""
    i = P2I[pos]
    return np.r_[LLR[pos - 1, AAS.index(mt)], site_tol[i], PLDDT[i], INTERFACE[i], DEG[i] / DEG.max(),
                 KD[mt] - KD[SEQ[pos - 1]], emb[i]]

data_v = labeled[labeled.pos.isin(P2I)].reset_index(drop=True)
print(f"varianti etichettate nella proteina matura: {len(data_v)} "
      f"(escluse {len(labeled) - len(data_v)} nel peptide segnale o non risolte)")
print(data_v.groupby(['fonte', 'label']).size())

In [ ]:
import matplotlib.pyplot as plt
import torch.nn as nn, torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_add_pool
from torch_geometric.utils import to_dense_batch, softmax as pyg_softmax

EI = torch.tensor(EDGE_INDEX, dtype=torch.long)
def make_graph(pos, mt, y=0.0):
    mask = torch.zeros(N_NODES, dtype=torch.bool); mask[P2I[pos]] = True
    return Data(x=torch.tensor(variant_x(pos, mt)), edge_index=EI, y=torch.tensor([float(y)]), mut=mask)

class Layer(nn.Module):
    """Ricetta GPS: GCN sul grafo dei contatti (locale) + multi-head attention su tutti i residui (globale)"""
    def __init__(self, d, heads=4, dropout=0.2, use_attn=True):
        super().__init__()
        self.local = GCNConv(d, d)
        self.attn = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True) if use_attn else None
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 2 * d), nn.ReLU(), nn.Dropout(dropout), nn.Linear(2 * d, d))
        self.drop = nn.Dropout(dropout)
    def forward(self, h, edge_index, batch):
        out = h + self.drop(F.relu(self.local(h, edge_index)))
        if self.attn is not None:
            hd, mask = to_dense_batch(h, batch)
            a, _ = self.attn(hd, hd, hd, key_padding_mask=~mask, need_weights=False)
            out = out + self.drop(a[mask])
        out = self.n1(out)
        return self.n2(out + self.mlp(out))

class VariantGT(nn.Module):
    """Il residuo mutato 'interroga' la proteina: readout = attenzione dal nodo mutato verso tutti i residui"""
    def __init__(self, in_dim, d=64, heads=4, layers=2, dropout=0.2, use_attn=True):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(in_dim, d), nn.ReLU(), nn.Dropout(dropout))
        self.layers = nn.ModuleList([Layer(d, heads, dropout, use_attn) for _ in range(layers)])
        self.q, self.k = nn.Linear(d, d), nn.Linear(d, d)
        self.head = nn.Sequential(nn.Linear(2 * d, d), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d, 1))
    def forward(self, data, return_att=False):
        h = self.inp(data.x)
        for l in self.layers:
            h = l(h, data.edge_index, data.batch)
        h_mut = h[data.mut]                                                  # (B, d): un nodo mutato per grafo
        e = (self.k(h) * self.q(h_mut)[data.batch]).sum(-1) / h.shape[1] ** 0.5
        a = pyg_softmax(e, data.batch)
        z = global_add_pool(a.unsqueeze(-1) * h, data.batch)
        logit = self.head(torch.cat([h_mut, z], -1)).squeeze(-1)
        return (logit, a) if return_att else logit

def train_gt(make, graphs, epochs=60, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    model = make().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    pos = sum(float(g.y) for g in graphs)
    pw = torch.tensor([(len(graphs) - pos) / max(pos, 1)], device=device)
    for _ in range(epochs):
        model.train()
        for b in DataLoader(graphs, batch_size=16, shuffle=True):
            b = b.to(device)
            loss = F.binary_cross_entropy_with_logits(model(b), b.y, pos_weight=pw)
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def predict_gt(model, graphs, bs=64):
    model.eval(); out = []
    with torch.no_grad():
        for b in DataLoader(graphs, batch_size=bs):
            out.append(torch.sigmoid(model(b.to(device))).cpu())
    return torch.cat(out).numpy()

## 1. Protocollo di valutazione comune
`valuta(oof, nome, traccia)` vuole un punteggio **out-of-fold** per ogni variante di `data_v` (stesso ordine). `folds()` restituisce i 5 split congelati.

In [ ]:
# ============ PROTOCOLLO DI VALUTAZIONE COMUNE (non modificare) ============
from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef
FOLD = data_v.fold.values.astype(int)
Y = data_v.label.values.astype(int)
LEADERBOARD = "risultati.csv"

def folds():
    """Genera (train_idx, test_idx) sui 5 fold congelati (split per POSIZIONE)."""
    for f in range(5):
        yield np.where(FOLD != f)[0], np.where(FOLD == f)[0]

def valuta(oof, nome, traccia='baseline', B=1000, seed=0):
    """oof: punteggio out-of-fold per ogni variante di data_v (più alto = più patogenica)."""
    oof = np.asarray(oof, dtype=float)
    rng = np.random.default_rng(seed)
    boot = []
    for _ in range(B):
        ix = rng.integers(0, len(Y), len(Y))
        if len(np.unique(Y[ix])) == 2:
            boot.append(roc_auc_score(Y[ix], oof[ix]))
    thr = 0.5 if (oof.min() >= 0 and oof.max() <= 1) else np.median(oof)
    r = dict(traccia=traccia, modello=nome, n=len(Y), AUROC=roc_auc_score(Y, oof),
             AUROC_lo=np.percentile(boot, 2.5), AUROC_hi=np.percentile(boot, 97.5),
             AUPRC=average_precision_score(Y, oof), MCC=matthews_corrcoef(Y, (oof > thr).astype(int)),
             data=time.strftime('%Y-%m-%d %H:%M'))
    pd.DataFrame([r]).to_csv(LEADERBOARD, mode='a', header=not os.path.exists(LEADERBOARD), index=False)
    print(f"{nome:40s} AUROC {r['AUROC']:.3f} [{r['AUROC_lo']:.3f}–{r['AUROC_hi']:.3f}]  "
          f"AUPRC {r['AUPRC']:.3f}  MCC {r['MCC']:.3f}")
    return r

def cv_gt(make_model, graph_list, epochs=60, **kw):
    oof = np.zeros(len(graph_list))
    for f, (tr, te) in enumerate(folds()):
        m = train_gt(make_model, [graph_list[i] for i in tr], epochs=epochs, seed=f, **kw)
        oof[te] = predict_gt(m, [graph_list[i] for i in te])
    return oof

## 2. Baseline di riferimento

In [ ]:
# ============ BASELINE DI RIFERIMENTO (lezione 4): ogni traccia deve confrontarsi con queste ============
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
esm_score = np.array([-LLR[p - 1, AAS.index(m)] for p, m in zip(data_v.pos, data_v.mt)])
Xtab = np.stack([tab_features(p, m) for p, m in zip(data_v.pos, data_v.mt)])
oof_lr = np.zeros(len(Y))
for tr, te in folds():
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced', C=0.3))
    oof_lr[te] = lr.fit(Xtab[tr], Y[tr]).predict_proba(Xtab[te])[:, 1]

graphs = [make_graph(p, m, y) for p, m, y in zip(data_v.pos, data_v.mt, data_v.label)]
in_dim = graphs[0].x.shape[1]
oof_gt = cv_gt(lambda: VariantGT(in_dim), graphs)

valuta(esm_score, 'ESM-2 zero-shot')
valuta(oof_lr, 'Reg. logistica (sito mutato)')
valuta(oof_gt, 'Graph Transformer L4 (monomero)')

---
# Traccia A — Il tetramero
Il monomero della lezione 4 vede l'interfaccia solo come un *flag*. Qui il grafo è l'**assemblaggio biologico** (PDB 1F41, 4 catene): ogni residuo esiste in 4 copie, gli archi hanno 3 tipi (intra-catena, monomero–monomero, dimero–dimero) e l'attenzione globale ha un **bias per tipo di arco** (come nella lezione 2).

In [ ]:
# ============ TRACCIA A — Il tetramero ============
def parse_assembly(text, n_chains=4):
    """Atomi per catena dell'assemblaggio biologico. Le copie simmetriche stanno in MODEL diversi:
    la chiave di catena è (modello, catena). Ritorna dict {(m, c): list di righe ATOM}."""
    model, chains = 0, {}
    for l in text.splitlines():
        if l.startswith('MODEL'):
            model += 1
        elif l.startswith('ATOM') and l[16] in ' A':
            chains.setdefault((max(model, 1), l[21]), []).append(l)
    keys = sorted(chains, key=lambda k: (k[0], k[1]))[:n_chains]
    return {k: chains[k] for k in keys}

def ca_from_lines(lines):
    return {int(l[22:26]): np.array([float(l[30:38]), float(l[38:46]), float(l[46:54])])
            for l in lines if l[12:16].strip() == 'CA'}

ASSEMBLY_TEXT = get("https://files.rcsb.org/download/1F41.pdb1").decode()
CH = parse_assembly(ASSEMBLY_TEXT)
CHAIN_KEYS = list(CH)
print("catene dell'assemblaggio (modello, catena):", CHAIN_KEYS)
CA4 = {k: ca_from_lines(v) for k, v in CH.items()}
# residui (numerazione MATURA nel PDB) presenti in tutte e 4 le catene
COMMON = sorted(set.intersection(*[set(c) for c in CA4.values()]))
POS4 = np.array([r + SIGNAL for r in COMMON])            # -> numerazione precursore (HGVS)
print(f"residui risolti in tutte le catene: {len(COMMON)} (maturo {COMMON[0]}–{COMMON[-1]})")

In [ ]:
# Nodi: 4 catene x residui comuni. Tipi di arco: 0 intra-catena, 1 monomero–monomero (stesso dimero),
# 2 dimero–dimero (interfaccia debole, canale della tiroxina)
ETYPES_A = ['intra', 'monomero-monomero', 'dimero-dimero']
nodes4 = [(k, r) for k in CHAIN_KEYS for r in COMMON]
XYZ4 = np.stack([CA4[k][r] for k, r in nodes4])
D4 = np.linalg.norm(XYZ4[:, None] - XYZ4[None], axis=-1)
model_of = np.array([k[0] for k, _ in nodes4]); chain_of = np.array([CHAIN_KEYS.index(k) for k, _ in nodes4])

# dimero = modello se le copie simmetriche sono in MODEL diversi; altrimenti catene (0,1) e (2,3)
dim = model_of if len(set(model_of)) > 1 else chain_of // 2
src, dst = np.nonzero((D4 < CUTOFF) & ~np.eye(len(nodes4), dtype=bool))
etype = np.where(chain_of[src] == chain_of[dst], 0, np.where(dim[src] == dim[dst], 1, 2))
EI4, ET4 = torch.tensor(np.stack([src, dst])), torch.tensor(etype)
print({ETYPES_A[t]: int((etype == t).sum()) for t in range(3)}, "archi orientati")

# Feature dei nodi: le stesse del monomero (L4), replicate sulle 4 catene + one-hot della catena
idx_mono = np.array([P2I.get(p, -1) for p in POS4])
assert (idx_mono >= 0).all(), "residui del cristallo non presenti nel modello AlphaFold"
BASE4 = np.hstack([np.tile(NODE_BASE[idx_mono], (4, 1)), np.repeat(np.eye(4), len(COMMON), 0)]).astype(np.float32)
P2I4 = {p: i for i, p in enumerate(POS4)}

def variant_x4(pos, mt, copies=(0, 1, 2, 3)):
    """La variante in 'copies' catene (tutte = omozigote; un sottoinsieme = tetramero misto)."""
    v = np.zeros((len(nodes4), 23), dtype=np.float32)
    for c in copies:
        i = c * len(COMMON) + P2I4[pos]
        v[i, 0] = 1; v[i, 1 + AAS.index(mt)] = 1
        v[i, 21] = LLR[pos - 1, AAS.index(mt)] / 5
        v[i, 22] = (KD[mt] - KD[SEQ[pos - 1]]) / 5
    return np.hstack([BASE4, v])

def make_graph4(pos, mt, y=0.0, copies=(0, 1, 2, 3)):
    x = variant_x4(pos, mt, copies)
    return Data(x=torch.tensor(x), edge_index=EI4, etype=ET4, y=torch.tensor([float(y)]),
                mut=torch.tensor(x[:, BASE4.shape[1]] > 0))

in_A = data_v.pos.isin(P2I4).values
print(f"varianti valutabili sul tetramero: {in_A.sum()} su {len(data_v)} "
      "(le altre cadono in residui non risolti nel cristallo)")

In [ ]:
from torch_geometric.utils import to_dense_adj
from torch_geometric.nn import global_mean_pool

class EdgeTypeAttention(nn.Module):
    """Attenzione globale con bias appreso per tipo di arco (+ 'nessun arco'), come nella lezione 2."""
    def __init__(self, d, heads, n_types, dropout=0.1):
        super().__init__()
        self.h, self.dh, self.T = heads, d // heads, n_types
        self.qkv, self.out = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.bias = nn.Linear(n_types + 1, heads, bias=False)
        self.drop = nn.Dropout(dropout)
    def forward(self, h, edge_index, etype, batch):
        hd, mask = to_dense_batch(h, batch); B, N, d = hd.shape
        q, k, v = self.qkv(hd).view(B, N, 3, self.h, self.dh).permute(2, 0, 3, 1, 4)
        E = to_dense_adj(edge_index, batch, edge_attr=F.one_hot(etype, self.T).float(),
                         max_num_nodes=N, batch_size=B).clamp(max=1)
        E = torch.cat([E, 1 - E.sum(-1, keepdim=True).clamp(max=1)], -1)
        s = q @ k.transpose(-1, -2) / self.dh ** 0.5 + self.bias(E).permute(0, 3, 1, 2)
        att = s.masked_fill(~mask[:, None, None, :], float('-inf')).softmax(-1)
        return self.out((self.drop(att) @ v).transpose(1, 2).reshape(B, N, d))[mask]

class TetraLayer(nn.Module):
    def __init__(self, d, heads, n_types, dropout=0.2):
        super().__init__()
        self.local = GCNConv(d, d)
        self.glob = EdgeTypeAttention(d, heads, n_types, dropout)
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 2 * d), nn.ReLU(), nn.Dropout(dropout), nn.Linear(2 * d, d))
    def forward(self, h, ei, et, batch):
        out = self.n1(h + F.relu(self.local(h, ei)) + self.glob(h, ei, et, batch))
        return self.n2(out + self.mlp(out))

class TetraGT(nn.Module):
    def __init__(self, in_dim, d=64, heads=4, layers=2, n_types=3, dropout=0.2):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(in_dim, d), nn.ReLU(), nn.Dropout(dropout))
        self.layers = nn.ModuleList([TetraLayer(d, heads, n_types, dropout) for _ in range(layers)])
        self.q, self.k = nn.Linear(d, d), nn.Linear(d, d)
        self.head = nn.Sequential(nn.Linear(2 * d, d), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d, 1))
    def forward(self, data):
        h = self.inp(data.x)
        for l in self.layers:
            h = l(h, data.edge_index, data.etype, data.batch)
        h_mut = global_mean_pool(h[data.mut], data.batch[data.mut])     # media delle copie mutate
        a = pyg_softmax((self.k(h) * self.q(h_mut)[data.batch]).sum(-1) / h.shape[1] ** 0.5, data.batch)
        z = global_add_pool(a.unsqueeze(-1) * h, data.batch)
        return self.head(torch.cat([h_mut, z], -1)).squeeze(-1)

# --- Esperimento minimo (fa girare la pipeline; il lavoro vero è nelle domande sotto) ---
sub = np.where(in_A)[0]
g4 = [make_graph4(data_v.pos[i], data_v.mt[i], data_v.label[i]) for i in sub]
oof_A = np.array(oof_gt, copy=True)          # varianti non valutabili: si usa la previsione L4
for f, (tr, te) in enumerate(folds()):
    tr_set, te_set = set(tr), set(te)
    tr_s, te_s = [j for j, i in enumerate(sub) if i in tr_set], [j for j, i in enumerate(sub) if i in te_set]
    m = train_gt(lambda: TetraGT(g4[0].x.shape[1]), [g4[j] for j in tr_s], epochs=40, seed=f)
    oof_A[sub[te_s]] = predict_gt(m, [g4[j] for j in te_s], bs=8)
valuta(oof_A, 'A: TetraGT (omozigote, bias tipo-arco)', traccia='A')

### Domande e attività della traccia A
1. **Il bias per tipo di arco serve?** Confrontate con un modello senza bias (attenzione semplice) e senza attenzione globale.
2. **Omozigote vs. eterozigote.** I pazienti ATTRv sono quasi sempre **eterozigoti**: i tetrameri sono misti. Allenate con `copies` casuali (1–3 catene mutate) come *data augmentation* e testate su varie composizioni. Cosa cambia?
3. **Lettura dei bias.** Stampate i pesi di `EdgeTypeAttention.bias` per tipo di arco: il modello dà importanza all'interfaccia dimero–dimero (dove agiscono gli stabilizzatori)?
4. **Costo.** Misurate tempo e memoria rispetto al monomero. Ne vale la pena?

---
# Traccia B — Fisica + apprendimento
Aggiungiamo ai nodi feature **fisiche** calcolabili senza simulazioni costose, più uno spazio per le feature di **dinamica molecolare** (fornite dal docente o prodotte dal gruppo).

In [ ]:
# ============ TRACCIA B — Fisica + apprendimento ============
from scipy.spatial import cKDTree

def gnm_fluct(xyz, cutoff=7.3):
    """Gaussian Network Model: fluttuazioni quadratiche medie per residuo (Bahar et al., 1997)."""
    d = np.linalg.norm(xyz[:, None] - xyz[None], axis=-1)
    K = -((d < cutoff) & (d > 0)).astype(float)
    K[np.diag_indices_from(K)] = -K.sum(1)
    w, v = np.linalg.eigh(K)
    nz = w > 1e-6
    return (v[:, nz] ** 2 / w[nz]).sum(1)

RADII = {'C': 1.7, 'N': 1.55, 'O': 1.52, 'S': 1.8}
MAX_ASA = dict(A=129, R=274, N=195, D=193, C=167, Q=225, E=223, G=104, H=224, I=197, L=201, K=236,
               M=224, F=240, P=159, S=155, T=172, W=285, Y=263, V=174)          # Tien et al. 2013

def sasa_per_residue(lines, n_points=100, probe=1.4):
    """Shrake–Rupley essenziale. lines: righe ATOM. Ritorna {(catena, resnum): Å^2}."""
    atoms = [l for l in lines if l[76:78].strip() != 'H' and l[12:16].strip()[0] != 'H']
    xyz = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])] for l in atoms])
    rad = np.array([RADII.get((l[76:78].strip() or l[12:16].strip()[0]).upper(), 1.8) for l in atoms]) + probe
    k = np.arange(n_points) + 0.5                                      # punti su sfera (spirale di Fibonacci)
    phi, th = np.arccos(1 - 2 * k / n_points), np.pi * (1 + 5 ** 0.5) * k
    sph = np.stack([np.cos(th) * np.sin(phi), np.sin(th) * np.sin(phi), np.cos(phi)], 1)
    tree = cKDTree(xyz); out = {}
    for i, l in enumerate(atoms):
        pts = xyz[i] + rad[i] * sph
        neigh = [j for j in tree.query_ball_point(xyz[i], rad[i] + rad.max()) if j != i]
        free = np.ones(n_points, bool)
        if neigh:
            dd = np.linalg.norm(pts[:, None] - xyz[neigh][None], axis=-1)
            free = (dd >= rad[neigh][None]).all(1)
        key = (l[21], int(l[22:26]))
        out[key] = out.get(key, 0.0) + 4 * np.pi * rad[i] ** 2 * free.mean()
    return out

# 1) Flessibilità GNM sul monomero AlphaFold e sul tetramero cristallografico
fl_mono = gnm_fluct(XYZ)
fl_tetra_all = gnm_fluct(XYZ4)
fl_tetra = np.full(N_NODES, np.nan)
for j, (k, r) in enumerate(nodes4[:len(COMMON)]):                    # catena 0 del tetramero
    fl_tetra[P2I[r + SIGNAL]] = fl_tetra_all[j]

# 2) Accessibilità relativa e superficie sepolta nell'assemblaggio (catena 0)
af_lines = [l for l in AF_TEXT.splitlines() if l.startswith('ATOM')]
sasa_af = sasa_per_residue(af_lines)
rsa = np.array([sasa_af.get(('A', p), np.nan) / MAX_ASA[SEQ[p - 1]] for p in POS])
tetra_lines = []
for ci, (k, lines) in enumerate(CH.items()):                           # rinomina catene: A, B, C, D
    tetra_lines += [l[:21] + 'ABCD'[ci] + l[22:] for l in lines]
sasa_t = sasa_per_residue(tetra_lines)
sasa_m = sasa_per_residue([l for l in tetra_lines if l[21] == 'A'])
buried = np.full(N_NODES, np.nan)
for r in COMMON:
    buried[P2I[r + SIGNAL]] = sasa_m.get(('A', r), 0) - sasa_t.get(('A', r), 0)

# 3) Distanza dal sito di clivaggio Lys48–Thr49 (maturo) = precursore 68–69 (Marcoux et al., 2015)
if SEQ[67] + SEQ[68] != 'KT':
    print("⚠️ controlla la numerazione: atteso K68-T69 nel precursore, trovato", SEQ[67:69])
mid = (XYZ[P2I[68]] + XYZ[P2I[69]]) / 2
d_cleav = np.linalg.norm(XYZ - mid, axis=1)

# 4) Distanza dal ligando nel canale della tiroxina (PDB 3TCT, TTR + tafamidis)
try:
    lig_txt = get("https://files.rcsb.org/download/3TCT.pdb").decode().splitlines()
    het = [l for l in lig_txt if l.startswith('HETATM') and l[17:20] not in ('HOH', 'NA ', 'CL ', 'SO4', 'GOL')]
    lig = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])] for l in het])
    ca3 = {int(l[22:26]): np.array([float(l[30:38]), float(l[38:46]), float(l[46:54])])
           for l in lig_txt if l.startswith('ATOM') and l[12:16].strip() == 'CA' and l[21] == 'A'}
    d_lig = np.array([np.min(np.linalg.norm(lig - ca3[p - SIGNAL], axis=1)) if (p - SIGNAL) in ca3 else np.nan
                      for p in POS])
except Exception as e:
    print("3TCT non disponibile:", e); d_lig = np.full(N_NODES, np.nan)

PHYS = pd.DataFrame(dict(pos=POS, gnm_mono=fl_mono, gnm_tetra=fl_tetra, rsa=rsa, sepolta_assemblaggio=buried,
                         dist_clivaggio=d_cleav, dist_ligando=d_lig))
display(PHYS.describe().T.round(2))

In [ ]:
# 5) Feature dalle simulazioni (fornite dal docente o prodotte dal gruppo)
#    md_residui.csv : pos (precursore), colonne numeriche per residuo (es. RMSF, frazione di apertura locale)
#    md_varianti.csv: pos, mt, colonne numeriche per variante (es. ΔG di apertura, ΔRMSF al sito di clivaggio)
for fname, kind in [('md_residui.csv', 'residui'), ('md_varianti.csv', 'varianti')]:
    if os.path.exists(fname):
        print("trovato", fname)
MD_RES = pd.read_csv('md_residui.csv') if os.path.exists('md_residui.csv') else None
MD_VAR = pd.read_csv('md_varianti.csv') if os.path.exists('md_varianti.csv') else None

phys_cols = ['gnm_mono', 'gnm_tetra', 'rsa', 'sepolta_assemblaggio', 'dist_clivaggio', 'dist_ligando']
P_res = PHYS.set_index('pos')[phys_cols]
if MD_RES is not None:
    P_res = P_res.join(MD_RES.set_index('pos'), how='left')
P_res = P_res.fillna(P_res.median())
NODE_BASE_B = np.hstack([NODE_BASE, z(P_res.loc[POS].values)]).astype(np.float32)
md_var_cols = [c for c in (MD_VAR.columns if MD_VAR is not None else []) if c not in ('pos', 'mt')]

def make_graph_B(pos, mt, y=0.0):
    i = P2I[pos]
    v = np.zeros((N_NODES, 23 + len(md_var_cols)), dtype=np.float32)
    v[i, 0] = 1; v[i, 1 + AAS.index(mt)] = 1
    v[i, 21] = LLR[pos - 1, AAS.index(mt)] / 5
    v[i, 22] = (KD[mt] - KD[SEQ[pos - 1]]) / 5
    if md_var_cols:
        row = MD_VAR[(MD_VAR.pos == pos) & (MD_VAR.mt == mt)]
        if len(row):
            v[i, 23:] = row[md_var_cols].values[0]
    mask = torch.zeros(N_NODES, dtype=torch.bool); mask[i] = True
    return Data(x=torch.tensor(np.hstack([NODE_BASE_B, v])), edge_index=EI, y=torch.tensor([float(y)]), mut=mask)

gB = [make_graph_B(p, m, y) for p, m, y in zip(data_v.pos, data_v.mt, data_v.label)]
oof_B = cv_gt(lambda: VariantGT(gB[0].x.shape[1]), gB)
valuta(oof_B, 'B: GT + feature fisiche' + (' + MD' if MD_RES is not None or md_var_cols else ''), traccia='B')

# Le feature fisiche da sole (sito mutato) in un modello lineare: quanto valgono senza il grafo?
Xphys = np.hstack([Xtab, P_res.loc[data_v.pos].values])
oof_Bl = np.zeros(len(Y))
for tr, te in folds():
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight='balanced', C=0.3))
    oof_Bl[te] = lr.fit(Xphys[tr], Y[tr]).predict_proba(Xphys[te])[:, 1]
valuta(oof_Bl, 'B: Reg. logistica + feature fisiche', traccia='B')

### Domande e attività della traccia B
1. **Quale feature fisica conta?** Ablazione una alla volta (GNM, RSA, superficie sepolta, distanza dal clivaggio, distanza dal ligando).
2. **Generalizzazione a siti nuovi.** Il guadagno (se c'è) è maggiore nei fold con meno varianti "vicine" a quelle di training? Misurate la distanza minima di ogni variante di test dai siti di training.
3. **Meccanismo.** Le varianti patogeniche sono più vicine al sito di clivaggio Lys48–Thr49 di quelle benigne? Test statistico + grafico.
4. **Dinamica molecolare.** Con `md_residui.csv` / `md_varianti.csv` (formato nella cella sopra) ripetete la valutazione. Documentate il protocollo di simulazione nel report.

---
# Traccia C — Più compiti: patogenicità + fenotipo
Le descrizioni delle varianti naturali di UniProt contengono spesso il **fenotipo** (polineuropatia, cardiomiopatia, forme oculari/leptomeningee). Costruiamo una bozza di etichette, la **curiamo**, e alleniamo un modello multi-task con loss **mascherata** (le etichette mancanti non contano).

In [ ]:
# ============ TRACCIA C — Più compiti: patogenicità + fenotipo ============
uni = json.loads(get("https://rest.uniprot.org/uniprotkb/P02766.json?fields=ft_variant"))
rows = []
for ft in uni.get('features', []):
    if ft.get('type') != 'Natural variant':
        continue
    alt = ft.get('alternativeSequence', {})
    wt, mts = alt.get('originalSequence', ''), alt.get('alternativeSequences', [])
    st = ft['location']['start']['value']
    if len(wt) == 1 and len(mts) == 1 and len(mts[0]) == 1:
        rows.append(dict(pos=st, wt=wt, mt=mts[0], descrizione=ft.get('description', '')))
uv = pd.DataFrame(rows)
uv = uv[[SEQ[p - 1] == w for p, w in zip(uv.pos, uv.wt)]]

PHENO = {'neuro':  r'neuropath|polyneuropath|autonomic',
         'cardio': r'cardiomyopath|cardiac|heart',
         'occhio': r'vitre|ocular|glaucom',
         'sistema_nervoso_centrale': r'leptomening|oculoleptomening|cns'}
d = uv.descrizione.str.lower()
for k, rx in PHENO.items():
    uv[k] = d.str.contains(rx).astype(float)
nessuno = uv[list(PHENO)].sum(1) == 0
uv.loc[nessuno, list(PHENO)] = np.nan                      # fenotipo non descritto = IGNOTO, non "assente"
uv['nome_classico'] = uv.wt + (uv.pos - SIGNAL).astype(str) + uv.mt
uv.to_csv('fenotipi_bozza.csv', index=False)
print(f"{len(uv)} varianti UniProt, con fenotipo descritto: {(~nessuno).sum()}")
display(uv[~nessuno][['nome_classico', 'descrizione'] + list(PHENO)].head(12))
print("➜ 'fenotipi_bozza.csv' è una BOZZA: il gruppo la verifica e la completa con la letteratura "
      "(e.g. registro delle mutazioni in amiloidosi ereditaria) e la ricarica come 'fenotipi_curati.csv'.")

In [ ]:
PH = pd.read_csv('fenotipi_curati.csv') if os.path.exists('fenotipi_curati.csv') else uv
AUX = ['neuro', 'cardio']                                   # compiti ausiliari (estendibili)
aux = data_v[['pos', 'wt', 'mt']].merge(PH[['pos', 'wt', 'mt'] + AUX], on=['pos', 'wt', 'mt'], how='left')[AUX].values
print("etichette ausiliarie disponibili:", {k: int(np.isfinite(aux[:, j]).sum()) for j, k in enumerate(AUX)})

class MultiTaskGT(VariantGT):
    def __init__(self, in_dim, n_aux=len(AUX), **kw):
        super().__init__(in_dim, **kw)
        d = self.q.in_features
        self.aux_heads = nn.Linear(2 * d, n_aux)
    def forward(self, data, return_aux=False):
        h = self.inp(data.x)
        for l in self.layers:
            h = l(h, data.edge_index, data.batch)
        h_mut = h[data.mut]
        a = pyg_softmax((self.k(h) * self.q(h_mut)[data.batch]).sum(-1) / h.shape[1] ** 0.5, data.batch)
        rep = torch.cat([h_mut, global_add_pool(a.unsqueeze(-1) * h, data.batch)], -1)
        main = self.head(rep).squeeze(-1)
        return (main, self.aux_heads(rep)) if return_aux else main

graphs_C = []
for g, a in zip(graphs, aux):
    g = g.clone(); g.aux = torch.tensor(np.nan_to_num(a, nan=0.0), dtype=torch.float32).view(1, -1)
    g.aux_mask = torch.tensor(np.isfinite(a), dtype=torch.float32).view(1, -1); graphs_C.append(g)

def train_mt(graph_list, lam=0.5, epochs=60, seed=0):
    torch.manual_seed(seed)
    model = MultiTaskGT(graph_list[0].x.shape[1]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
    pos = sum(float(g.y) for g in graph_list)
    pw = torch.tensor([(len(graph_list) - pos) / max(pos, 1)], device=device)
    for _ in range(epochs):
        model.train()
        for b in DataLoader(graph_list, batch_size=16, shuffle=True):
            b = b.to(device)
            main, aux_logit = model(b, return_aux=True)
            loss = F.binary_cross_entropy_with_logits(main, b.y, pos_weight=pw)
            la = F.binary_cross_entropy_with_logits(aux_logit, b.aux, reduction='none')
            loss = loss + lam * (la * b.aux_mask).sum() / b.aux_mask.sum().clamp(min=1)   # loss mascherata
            opt.zero_grad(); loss.backward(); opt.step()
    return model

oof_C = np.zeros(len(Y))
for f, (tr, te) in enumerate(folds()):
    m = train_mt([graphs_C[i] for i in tr], seed=f)
    oof_C[te] = predict_gt(m, [graphs[i] for i in te])
valuta(oof_C, 'C: GT multi-task (patog. + neuro/cardio)', traccia='C')

### Domande e attività della traccia C
1. **Curatela.** Verificate e completate `fenotipi_bozza.csv` con la letteratura (almeno 40 varianti con fenotipo verificato, fonte per ogni riga). Ricaricatelo come `fenotipi_curati.csv`.
2. **Il multi-task aiuta?** Variate `lam` (0, 0.25, 0.5, 1). Il compito principale migliora, peggiora o resta uguale?
3. **Predire il fenotipo.** Valutate le teste ausiliarie (AUROC neuro, cardio) con gli stessi fold, sulle sole varianti con etichetta. Discutete il numero di esempi.
4. **Età d'esordio (facoltativo).** Aggiungete una testa di regressione per l'età d'esordio media riportata in letteratura.

---
# Traccia D — VUS, AlphaMissense e calibrazione
Un punteggio è utile in clinica solo se è **calibrato** e traducibile in una **forza di evidenza** (ACMG/AMP: PP3 a favore della patogenicità, BP4 a favore della benignità).

In [ ]:
# ============ TRACCIA D — VUS, AlphaMissense e calibrazione ============
from sklearn.isotonic import IsotonicRegression
am = pd.read_csv(io.BytesIO(get(meta['amAnnotationsUrl'])))
am['wt'], am['pos'], am['mt'] = am.protein_variant.str[0], am.protein_variant.str[1:-1].astype(int), am.protein_variant.str[-1]
am_score = data_v[['pos', 'wt', 'mt']].merge(am[['pos', 'wt', 'mt', 'am_pathogenicity']], how='left').am_pathogenicity.values
print("varianti senza punteggio AlphaMissense:", int(np.isnan(am_score).sum()))
valuta(np.nan_to_num(am_score, nan=np.nanmedian(am_score)), 'D: AlphaMissense', traccia='D')

def local_LR(scores, B_prior=None):
    """Calibrazione cross-fittata: isotonica allenata sugli ALTRI fold -> posterior -> rapporto di verosimiglianza
    LR = odds(posterior) / odds(prevalenza del training)."""
    post = np.zeros(len(Y))
    for tr, te in folds():
        iso = IsotonicRegression(y_min=1e-3, y_max=1 - 1e-3, out_of_bounds='clip').fit(scores[tr], Y[tr])
        post[te] = iso.predict(scores[te])
    prev = Y.mean()
    return (post / (1 - post)) / (prev / (1 - prev)), post

# Soglie di forza dell'evidenza (Tavtigian et al., 2018; quadro bayesiano ACMG/AMP)
STRENGTH = [(350, 'molto forte'), (18.7, 'forte'), (4.33, 'moderata'), (2.08, 'di supporto')]
def strength(lr):
    for t, s in STRENGTH:
        if lr >= t: return 'PP3 ' + s
        if lr <= 1 / t: return 'BP4 ' + s
    return '—'

for nome, sc in [('ESM-2', esm_score), ('AlphaMissense', np.nan_to_num(am_score, nan=np.nanmedian(am_score))), ('GT L4', oof_gt)]:
    lr_, _ = local_LR(sc)
    lab = pd.Series([strength(v) for v in lr_]).value_counts()
    print(f"\n{nome}: distribuzione della forza di evidenza sulle varianti etichettate"); print(lab.to_string())

In [ ]:
# VUS: classifica con il GT (ensemble sui 5 fold) e confronto con AlphaMissense
vus = cv[cv.label.isna() & cv.pos.isin(P2I)].copy()
models_D = [train_gt(lambda: VariantGT(in_dim), [graphs[i] for i in tr], seed=f) for f, (tr, te) in enumerate(folds())]
gv = [make_graph(p, m) for p, m in zip(vus.pos, vus.mt)]
vus['P_GT'] = np.mean([predict_gt(m, gv) for m in models_D], 0)
vus = vus.merge(am[['pos', 'wt', 'mt', 'am_pathogenicity', 'am_class']], how='left')
vus['LLR_ESM'] = [LLR[p - 1, AAS.index(m)] for p, m in zip(vus.pos, vus.mt)]
vus['accordo'] = (vus.P_GT > 0.5) == (vus.am_pathogenicity > 0.5)
cols = ['nome_classico', 'nome_hgvs', 'classificazione', 'revisione', 'P_GT', 'am_pathogenicity', 'am_class', 'LLR_ESM', 'accordo']
vus.sort_values('P_GT', ascending=False)[cols].to_csv('vus_classifica.csv', index=False)
print(f"{len(vus)} VUS; accordo GT–AlphaMissense: {vus.accordo.mean():.0%}")
display(vus.sort_values('P_GT', ascending=False)[cols].head(10))
print("➜ i 10 casi in cima e i casi di DISACCORDO sono il materiale della revisione bibliografica (deliverable D).")

### Domande e attività della traccia D
1. **Confronto equo.** AlphaMissense è calibrato anche su frequenze di popolazione e il nostro set di benigne usa gnomAD: quanto è "cieco" il confronto? Ripetete escludendo le presunte benigne da gnomAD.
2. **Calibrazione.** Curve di calibrazione (cross-fittate) per GT, ESM-2 e AlphaMissense. Quale modello raggiunge evidenza *moderata* o *forte* su più varianti?
3. **Soglie.** Le soglie usate (rapporti di verosimiglianza 2.08 / 4.33 / 18.7 / 350) derivano da Tavtigian et al. (2018) con prior 0.10. Pejaver et al. (2022) usano un prior diverso: ricalcolate e discutete.
4. **Revisione delle VUS.** Per le prime 10 VUS di `vus_classifica.csv` e per i casi di disaccordo GT–AlphaMissense, cercate in letteratura dati funzionali o clinici. Il modello aveva ragione?

---
## Classifica del gruppo

In [ ]:
# ============ CLASSIFICA DEL GRUPPO ============
lb = pd.read_csv(LEADERBOARD)
display(lb.drop_duplicates(['traccia', 'modello'], keep='last').sort_values('AUROC', ascending=False).round(3))
print("Consegnare 'risultati.csv' insieme al notebook: il docente unisce le classifiche dei gruppi.")